# vLLM inference optimizations on a Kaggle T4

Question: how much does each vLLM serving optimization change latency and throughput when you turn it on or off?

Setup: one Tesla T4 (16 GB, Turing, no bf16, so fp16), `Qwen/Qwen2.5-1.5B-Instruct`, the vLLM OpenAI-compatible server. A small load generator (Python standard library only) measures each effect. Each server config starts, runs its workloads, and stops.

Terms used below:

- **Prefill**: the phase that reads the whole prompt and builds the KV cache. It is compute-bound.
- **Decode**: the phase that makes one token per step. It is memory-bandwidth-bound.
- **TTFT** (time to first token): the wait from request start to the first output token.
- **TPOT** (time per output token): the average time between output tokens after the first.
- **Prefix caching**: vLLM keeps the KV cache of earlier prompts and reuses it when a new prompt starts with the same text.
- **Speculative decoding**: a cheap method guesses several next tokens. The model checks all guesses in one step. Here the guess method is n-gram lookup in the prompt.
- **Chunked prefill**: vLLM splits a long prompt into pieces and mixes them with decode work in the same step. `--max-num-batched-tokens` sets the token budget of one step.
- **Request queuing**: a request waits when the server already runs its maximum number of sequences (`--max-num-seqs`).

## Predict before running

Write your own guess first. Then compare it with the table at the end.

| Optimization | Workload | Expected direction |
|---|---|---|
| Prefix caching on | 64 requests share a long system text | TTFT falls. The server skips prefill for the shared part. |
| Speculative decoding on | Copy a passage, concurrency 1 and 16 | TPOT falls at concurrency 1. The gain shrinks or reverses at concurrency 16, because the GPU is already busy with real work. |
| Smaller token budget per step (512) | 8 short streams plus 8 long prompts | Short-stream TPOT p99 improves, because a long prefill stalls decode less. Long-prompt TTFT rises. |
| Low sequence limit (4) | 64 requests at concurrency 32 | TTFT median and p99 rise a lot, because requests queue. TPOT median falls, because each step serves fewer sequences. |
| Streaming | 8 requests, concurrency 1 | The user sees the first token long before a non-streaming reply returns. |

In [ ]:
# 1. Install vLLM (into the Kaggle VM, not your laptop)
!pip install -q vllm 2>&1 | tail -3
!python -c "import vllm; print('vllm', vllm.__version__)"
!nvidia-smi --query-gpu=index,name,memory.total,compute_cap --format=csv

In [ ]:
# 2. Helpers: server control, load generator, prompt builder
# Do not import torch with CUDA in this kernel. It would hold GPU memory that vLLM needs.
import json, os, random, statistics, subprocess, time, urllib.request
from concurrent.futures import ThreadPoolExecutor

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE = "http://localhost:8000"
MAX_LEN, GPU_UTIL = 4096, 0.9
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RESULTS, FAILED = {}, {}   # RESULTS[(config, workload)] = summary dict

# ---- GPU and server control ----
def gpu_used_mib():
    out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
    return [int(x) for x in out.split()]

def start_server(name, extra_args=()):
    log = f"{OUT_DIR}/{name}_server.log"
    cmd = ["vllm", "serve", MODEL, "--dtype", "half", "--max-model-len", str(MAX_LEN),
           "--gpu-memory-utilization", str(GPU_UTIL), "--port", "8000", *extra_args]
    print(name, "->", " ".join(c if " " not in c and "{" not in c else repr(c) for c in cmd[2:]))
    srv = subprocess.Popen(cmd, stdout=open(log, "w"), stderr=subprocess.STDOUT,
                           env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
    srv.log = log
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen(f"{BASE}/health"); break
        except Exception:
            if srv.poll() is not None:
                raise RuntimeError("server exited, see log tail")
            if time.time() - t0 > 1200:
                srv.kill()
                raise RuntimeError("server not ready after 20 min")
            time.sleep(2)
    print(f"{name}: server ready after {time.time()-t0:.0f} s")
    return srv

def stop_server(srv):
    srv.terminate()
    try: srv.wait(timeout=90)
    except subprocess.TimeoutExpired: srv.kill(); srv.wait()
    for _ in range(60):   # wait until the GPU memory is free again
        if max(gpu_used_mib()) < 500: return
        time.sleep(2)
    print("warning: GPU memory not free after 120 s:", gpu_used_mib())

def get_metrics(substr):
    text = urllib.request.urlopen(f"{BASE}/metrics", timeout=30).read().decode()
    return [l for l in text.splitlines() if substr in l and not l.startswith("#")]

# ---- Prompt builder: fixed word list, seeded, sized with the server's /tokenize ----
WORDS = ("time year people way day man thing woman life child world school state family student group country "
         "problem hand part place case week company system program question work government number night point "
         "home water room mother area money story fact month lot right study book eye job word business issue "
         "side kind head house service friend father power hour game line end member law car city community "
         "name president team minute idea kid body information back parent face others level office door health "
         "person art war history party result change morning reason research girl guy moment air teacher force "
         "education").split()

def count_tokens(text):
    body = json.dumps({"model": MODEL, "prompt": text}).encode()
    req = urllib.request.Request(f"{BASE}/tokenize", body, {"Content-Type": "application/json"})
    return json.load(urllib.request.urlopen(req, timeout=30))["count"]

_TOK_PER_WORD = {}
def tokens_per_word():
    key = BASE
    if key not in _TOK_PER_WORD:
        sample = " ".join(random.Random(0).choices(WORDS, k=400))
        try:
            _TOK_PER_WORD[key] = count_tokens(sample) / 400
            print(f"/tokenize works: {_TOK_PER_WORD[key]:.2f} tokens per filler word")
        except Exception as e:
            _TOK_PER_WORD[key] = 1 / 0.75
            print(f"/tokenize failed ({e!r}); fallback 0.75 words per token")
    return _TOK_PER_WORD[key]

def make_text(n_tokens, seed):
    n_words = max(1, round(n_tokens / tokens_per_word()))
    return " ".join(random.Random(seed).choices(WORDS, k=n_words))

# ---- Load generator ----
def one_request(prompt, max_tokens, stream=True):
    body = {"model": MODEL, "prompt": prompt, "max_tokens": max_tokens, "temperature": 0, "ignore_eos": True}
    if stream:
        body.update(stream=True, stream_options={"include_usage": True})
    req = urllib.request.Request(f"{BASE}/v1/completions", json.dumps(body).encode(), {"Content-Type": "application/json"})
    t0 = time.perf_counter(); first = last = None; tokens = 0
    try:
        with urllib.request.urlopen(req, timeout=900) as r:
            if not stream:
                tokens = json.load(r)["usage"]["completion_tokens"]
            else:
                for raw in r:
                    line = raw.decode().strip()
                    if not line.startswith("data:"): continue
                    data = line[5:].strip()
                    if data == "[DONE]": break
                    ev = json.loads(data)
                    ch = ev.get("choices") or []
                    if ch and ch[0].get("text"):
                        last = time.perf_counter()
                        if first is None: first = last
                    if ev.get("usage"): tokens = ev["usage"]["completion_tokens"]
    except Exception as e:
        return {"error": repr(e)}
    end = time.perf_counter()
    res = {"latency": end - t0, "tokens": tokens}
    if stream and first is not None:
        res["ttft"] = first - t0
        if tokens > 1 and last > first:
            res["tpot"] = (last - first) / (tokens - 1)   # last = time of the last text chunk
    return res

def run_load(reqs, concurrency):
    """reqs: list of dicts {prompt, max_tokens, stream=True, delay=0, tag=''}. Returns (results, wall seconds)."""
    t_run = time.perf_counter()
    def job(r):
        wait = r.get("delay", 0) - (time.perf_counter() - t_run)
        if wait > 0: time.sleep(wait)
        out = one_request(r["prompt"], r["max_tokens"], r.get("stream", True))
        out["tag"] = r.get("tag", "")
        return out
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        results = list(ex.map(job, reqs))
    return results, time.perf_counter() - t_run

def pct(xs, p):
    xs = sorted(xs)
    if not xs: return float("nan")
    k = (len(xs) - 1) * p / 100; f = int(k); c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)

def summarize(results, wall, tag=None):
    rs = [r for r in results if "error" not in r and (tag is None or r["tag"] == tag)]
    errs = [r["error"] for r in results if "error" in r and (tag is None or r["tag"] == tag)]
    ttft = [r["ttft"] * 1000 for r in rs if "ttft" in r]
    tpot = [r["tpot"] * 1000 for r in rs if "tpot" in r]
    lat = [r["latency"] * 1000 for r in rs]
    return {"n": len(rs), "errors": len(errs),
            "ttft_med_ms": statistics.median(ttft) if ttft else float("nan"), "ttft_p99_ms": pct(ttft, 99),
            "tpot_med_ms": statistics.median(tpot) if tpot else float("nan"), "tpot_p99_ms": pct(tpot, 99),
            "latency_med_ms": statistics.median(lat) if lat else float("nan"),
            "tok_s": sum(r["tokens"] for r in rs) / wall}

def record(cfg, workload, summary):
    RESULTS[(cfg, workload)] = summary
    print(f"[{cfg} | {workload}] " + ", ".join(f"{k}={v:.1f}" if isinstance(v, float) else f"{k}={v}" for k, v in summary.items()))

# ---- Workloads ----
def warmup():
    one_request("Hello", 8)

def wl_prefix(cfg):
    shared = make_text(2000, seed=1)
    reqs = [{"prompt": f"{shared}\n\nQuestion {i}: {make_text(12, seed=100 + i)}?\nAnswer:", "max_tokens": 32} for i in range(64)]
    one_request(reqs[0]["prompt"], 32)   # warm-up: puts the shared prefix in the cache (when caching is on)
    before = get_metrics("prefix_cache")
    results, wall = run_load(reqs[1:] + reqs[:1], concurrency=8)
    after = get_metrics("prefix_cache")
    print("prefix_cache metrics before:", *before, sep="\n  "); print("prefix_cache metrics after:", *after, sep="\n  ")
    record(cfg, "prefix", summarize(results, wall))

def wl_spec(cfg):
    for c in (1, 16):
        reqs = [{"prompt": f"Copy the text below exactly:\n\n{make_text(500, seed=200 + i)}\n\nExact copy:\n", "max_tokens": 256} for i in range(16)]
        results, wall = run_load(reqs, concurrency=c)
        record(cfg, f"spec_c{c}", summarize(results, wall))
    print("spec_decode metrics:", *get_metrics("spec_decode"), sep="\n  ")

def wl_mixed(cfg):
    short = [{"prompt": make_text(100, seed=300 + i), "max_tokens": 256, "tag": "short"} for i in range(8)]
    long_ = [{"prompt": make_text(3000, seed=400 + i), "max_tokens": 16, "tag": "long", "delay": 1.0 + 0.5 * i} for i in range(8)]
    results, wall = run_load(short + long_, concurrency=16)
    record(cfg, "mixed_short", summarize(results, wall, "short"))
    record(cfg, "mixed_long", summarize(results, wall, "long"))

def wl_queue(cfg):
    reqs = [{"prompt": make_text(256, seed=500 + i), "max_tokens": 128} for i in range(64)]
    results, wall = run_load(reqs, concurrency=32)
    record(cfg, "queue", summarize(results, wall))

def wl_stream(cfg):
    mk = lambda base, stream: [{"prompt": make_text(100, seed=base + i), "max_tokens": 256, "stream": stream} for i in range(8)]
    results, wall = run_load(mk(600, True), concurrency=1); record(cfg, "stream", summarize(results, wall))
    results, wall = run_load(mk(700, False), concurrency=1); record(cfg, "nostream", summarize(results, wall))

def run_config(name, extra_args, workloads):
    srv = None
    try:
        srv = start_server(name, extra_args)
        warmup()
        for wl in workloads: wl(name)
    except Exception as e:
        FAILED[name] = repr(e)
        print(f"!! {name} failed: {e!r}")
        if srv is not None or os.path.exists(f"{OUT_DIR}/{name}_server.log"):
            print(open(f"{OUT_DIR}/{name}_server.log").read()[-3000:])
        RESULTS[(name, "failed")] = {"reason": repr(e)}
    finally:
        if srv is not None: stop_server(srv)

In [ ]:
# 3. S1: baseline (vLLM defaults). Runs every workload once.
run_config("S1", [], [wl_prefix, wl_spec, wl_mixed, wl_queue, wl_stream])

In [ ]:
# 4. S2: prefix caching off
run_config("S2", ["--no-enable-prefix-caching"], [wl_prefix])

In [ ]:
# 5. S3: speculative decoding on (n-gram lookup, 4 guessed tokens)
SPEC = json.dumps({"method": "ngram", "num_speculative_tokens": 4, "prompt_lookup_max": 4, "prompt_lookup_min": 2})
run_config("S3", ["--speculative-config", SPEC], [wl_spec])

In [ ]:
# 6. S4: token budget per step cut to 512
run_config("S4", ["--max-num-batched-tokens", "512"], [wl_mixed])

In [ ]:
# 7. S5: at most 4 sequences at once
run_config("S5", ["--max-num-seqs", "4"], [wl_queue])

In [ ]:
# 8. Comparison table
import pandas as pd
def g(cfg, wl, key):
    return RESULTS.get((cfg, wl), {}).get(key, float("nan"))
rows = [
    # (optimization, workload, metric, off, on)
    ("Prefix caching", "64 requests, shared 2k-token prefix, c=8", "TTFT median ms", g("S2", "prefix", "ttft_med_ms"), g("S1", "prefix", "ttft_med_ms")),
    ("Prefix caching", "64 requests, shared 2k-token prefix, c=8", "TTFT p99 ms", g("S2", "prefix", "ttft_p99_ms"), g("S1", "prefix", "ttft_p99_ms")),
    ("Speculative decoding", "copy 500-token passage, c=1", "TPOT median ms", g("S1", "spec_c1", "tpot_med_ms"), g("S3", "spec_c1", "tpot_med_ms")),
    ("Speculative decoding", "copy 500-token passage, c=1", "output tok/s", g("S1", "spec_c1", "tok_s"), g("S3", "spec_c1", "tok_s")),
    ("Speculative decoding", "copy 500-token passage, c=16", "TPOT median ms", g("S1", "spec_c16", "tpot_med_ms"), g("S3", "spec_c16", "tpot_med_ms")),
    ("Speculative decoding", "copy 500-token passage, c=16", "output tok/s", g("S1", "spec_c16", "tok_s"), g("S3", "spec_c16", "tok_s")),
    ("Token budget 512 (chunked prefill)", "8 short streams + 8 long prompts, c=16", "short TPOT p99 ms", g("S1", "mixed_short", "tpot_p99_ms"), g("S4", "mixed_short", "tpot_p99_ms")),
    ("Token budget 512 (chunked prefill)", "8 short streams + 8 long prompts, c=16", "long TTFT median ms", g("S1", "mixed_long", "ttft_med_ms"), g("S4", "mixed_long", "ttft_med_ms")),
    ("Sequence limit 4 (request queuing)", "64 requests, c=32", "TTFT median ms", g("S1", "queue", "ttft_med_ms"), g("S5", "queue", "ttft_med_ms")),
    ("Sequence limit 4 (request queuing)", "64 requests, c=32", "TTFT p99 ms", g("S1", "queue", "ttft_p99_ms"), g("S5", "queue", "ttft_p99_ms")),
    ("Sequence limit 4 (request queuing)", "64 requests, c=32", "TPOT median ms", g("S1", "queue", "tpot_med_ms"), g("S5", "queue", "tpot_med_ms")),
    ("Streaming", "8 requests, c=1, 256 tokens", "wait before text ms (off: full reply, on: first token)", g("S1", "nostream", "latency_med_ms"), g("S1", "stream", "ttft_med_ms")),
]
df = pd.DataFrame(rows, columns=["optimization", "workload", "metric", "off", "on"])
df["change_pct"] = (df["on"] - df["off"]) / df["off"] * 100
df.round(1).to_csv(f"{OUT_DIR}/optimizations.csv", index=False)
print(df.round(1).to_markdown(index=False))
if FAILED: print("FAILED configs:", FAILED)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## How to read the table

- `off` is the baseline or the feature disabled. `on` is the feature enabled, or the changed setting. For the sequence limit, `off` is the vLLM default and `on` is the limit of 4. For streaming, `off` is the non-streaming reply time and `on` is the streaming TTFT.
- `change_pct` is negative when the number fell. Lower is better for latency rows. Higher is better for tok/s rows.
- Check the `/metrics` lines printed above. They show why a result moved: prefix cache hits, and accepted speculative tokens.
- A `nan` means that config failed. The log tail is printed in its cell.

## Exam rules this proves

1. **inference.2**: Prefill is compute-bound and sets TTFT. Decode is memory-bound and sets TPOT. Tune them with different knobs.
2. **inference.3**: Prefix caching removes repeated prefill work. It helps only when many prompts share the same start.
3. **inference.3**: Speculative decoding trades spare GPU compute for fewer decode steps. The gain shrinks when the batch is already large.
4. **inference.1**: A limit on running sequences turns extra load into queue time. Queue time shows up in TTFT, not TPOT.
5. **inference.5**: Chunked prefill limits how long one long prompt can stall decode for other users. Token streaming lets the user see text at TTFT instead of at the full reply time.